# Env test

In [1]:
from fy_project.env import P2PEnergyTrading
from fy_project.agent import P2PTradingPolicy

from ray.rllib.core.rl_module import RLModuleSpec

import json
import numpy as np
from pathlib import Path

c:\Users\Abhyuday Chauhan\.conda\envs\FY_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-07 20:35:09,506	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-07 20:35:17,775	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
path = Path(r"C:\Users\Abhyuday Chauhan\projects\FY_project\src\config")

In [3]:
with open(path.joinpath("env_config.json")) as f:
    env_cfg = json.load(f)
agent_cfg = env_cfg['agent_cfg']

In [4]:
env = P2PEnergyTrading(**env_cfg)

C:\Users\Abhyuday Chauhan\projects\FY_project\src\fy_project\IEX_data\iex_hourly_market.csv


c:\Users\Abhyuday Chauhan\.conda\envs\FY_project\Lib\site-packages\gymnasium\spaces\box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\Abhyuday Chauhan\.conda\envs\FY_project\Lib\site-packages\gymnasium\spaces\box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [5]:
obs, info = env.reset()

2025-03-14 00:00:00


In [6]:
obs['household_0']

{'market_price': array([0.39119804, 0.38548124, 0.3763133 , 0.36391735, 0.3689752 ,
        0.42926848, 0.73101383, 0.6684515 , 0.3497782 , 0.37095764,
        0.3208764 , 0.3028271 , 0.27828887, 0.25944802, 0.27548316,
        0.31486663, 0.35300723, 0.34398934, 0.6003099 , 0.84811074,
        0.48960403, 0.43204966, 0.42369536, 0.36648694], dtype=float32),
 'battery_soc': array([0.30692863], dtype=float32),
 'battery_capacity': array([7.8425813], dtype=float32),
 'forecast_demand': array([0.1433377 , 0.13105161, 0.12286089, 0.11467016, 0.12286089,
        0.22524497, 0.34810588, 0.36858267, 0.26619858, 0.20476815,
        0.18429133, 0.17200524, 0.16381453, 0.17200524, 0.18429133,
        0.22524497, 0.2866754 , 0.36858267, 0.4095363 , 0.40134558,
        0.34810588, 0.2866754 , 0.22524497, 0.18429133], dtype=float32),
 'generation': array([0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.02095853, 0.08899134, 0.16480082, 0.22985832,
        0.2766846

In [7]:
env.observation_spaces['household_0'].contains(obs['household_0'])

True

In [8]:
action = {"q_buy": np.random.random(size=(24,)).astype(dtype=np.float32)}
action = {"household_0" : action}

In [9]:
env.action_spaces['household_0'].contains(action['household_0'])

True

In [10]:
obs, rewards, terminated, _, _ = env.step(action)

In [11]:
env.observation_spaces['household_0'].contains(obs['household_0'])

True

In [12]:
model_spec = RLModuleSpec(
        module_class=P2PTradingPolicy, 
        model_config=agent_cfg,
        observation_space=env.observation_spaces['household_0'],
        action_space=env.action_spaces['household_0']
    )

policy = model_spec.build()

2026-03-07 20:35:23,146	WARNING rl_module.py:459 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!


In [13]:
import torch

In [14]:
test_input = {
    "market_price": torch.rand(1,24),
    "battery_soc": torch.rand(1, 1),
    "battery_capacity"  : torch.rand(1, 1),
    "forecast_demand": torch.rand(1, 24),
    "generation": torch.rand(1, 24)
}

In [15]:
outputs = policy.forward_inference(test_input)

torch.Size([1, 64]) torch.Size([1, 64]) torch.Size([1, 64]) torch.Size([1, 64])
torch.Size([1, 256])


In [16]:
outputs['action_dist_inputs'].shape

torch.Size([1, 24])